# Warpmate, Step 6b: 4-piece endgames on the GPU

This notebook:

1. Builds Warpmate and runs every test, including the new **GPU solver** test (GPU tables must match the CPU byte for byte).
2. Solves 4-piece endgames on the GPU: **KQvKR, KRvKR, KRvKB, KRvKN, KRvKP**, about 33.5 million positions each.
3. Benchmarks CPU vs GPU on KQvKR.
4. Checks the GPU-built tables against the official Syzygy tablebases.

**Before you start:** *Runtime > Change runtime type > T4 GPU > Save*, then run the cells top to bottom.

In [ ]:
!nvidia-smi

## Get the code

Drag `warpmate-colab.zip` into the **Files** panel on the left (or run the upload cell). If no zip is found, the next cell clones from GitHub instead.

In [ ]:
# Optional: upload warpmate-colab.zip from your computer.
from google.colab import files
files.upload()

In [ ]:
import os
os.chdir('/content')
if os.path.exists('/content/warpmate-colab.zip'):
    !mkdir -p Warpmate && unzip -o -q warpmate-colab.zip -d Warpmate
    print('Using the uploaded zip')
elif os.path.isdir('/content/Warpmate/.git'):
    # Already cloned in this session: update it instead of starting over, so a
    # re-run keeps earlier results (tables, dataset, trained model).
    !cd /content/Warpmate && git pull -q
    print('Updated the existing copy from GitHub')
else:
    !rm -rf Warpmate && git clone -q https://github.com/vasishtk1/Warpmate.git
    print('Cloned from GitHub')
%cd /content/Warpmate

## Build

Look for `Warpmate GPU support: ON`.

In [ ]:
import os
if os.path.exists('/usr/local/cuda/bin/nvcc'):
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -4
!cmake --build build -j 2>&1 | tail -6

## Tests

Two suites: `egtb` (the solver on the CPU) and **`egtb-gpu`** (new: every 3-piece table solved on both CPU and GPU must be identical).

In [ ]:
!ctest --test-dir build --output-on-failure

## Solve 4-piece endgames on the GPU

`KRvKP` depends on the other four 4-piece endgames (the pawn can promote to a queen, rook, bishop or knight), so this one command solves all five, plus the 3-piece ones they need. Each line shows the time, rounds, results and the longest forced win.

The known longest forced mate in **KQvKR is 35 moves**.

In [ ]:
!./build/warpmate-gen --gpu KRvKP 2>&1 | tee gen_gpu.log

Look up the longest KQvKR win from the GPU-built table and play it out:

In [ ]:
import re, subprocess
log = open('gen_gpu.log').read()
fen = re.search(r'^KQvKR .*?\n\s+longest win: mate in \d+, e\.g\. (.+)$', log, re.M).group(1)
print(subprocess.run(['./build/warpmate-probe', fen], capture_output=True, text=True).stdout)

## Benchmark: KQvKR, CPU vs GPU

Times only the KQvKR table itself (the 3-piece tables it needs are solved first, untimed). The CPU run uses both of Colab's CPU threads and **takes a few minutes**; the GPU runs 3 times after a warm-up. It also checks that the CPU and GPU tables are identical.

In [ ]:
!echo "CPU: $(lscpu | grep 'Model name' | sed 's/Model name: *//') ($(nproc) threads)"
!./build/warpmate-bench KQvKR --cpu-trials 1 --gpu-trials 3

## Check against Syzygy

Downloads the official win/draw/loss files (a few MB) and compares them with the **GPU-built** tables: every row of the 3-piece tables, and a random sample of 1,000,000 rows from each 4-piece table (checking all 33.5 million rows in Python would take 15+ minutes per table).

In [ ]:
!pip -q install chess
!mkdir -p syzygy
!for t in KQvK KRvK KBvK KNvK KPvK KQvKR KRvKR KRvKB KRvKN KRvKP; do wget -q -nc -P syzygy https://tablebase.lichess.ovh/tables/standard/3-4-5-wdl/$t.rtbw; done
!ls syzygy

In [ ]:
!python scripts/verify_syzygy.py --syzygy syzygy tables/KQvK.wmtb tables/KRvK.wmtb tables/KBvK.wmtb tables/KNvK.wmtb tables/KPvK.wmtb
!python scripts/verify_syzygy.py --syzygy syzygy --sample 1000000 tables/KQvKR.wmtb tables/KRvKR.wmtb tables/KRvKB.wmtb tables/KRvKN.wmtb tables/KRvKP.wmtb

## Done

Copy the output of the **Tests**, **Solve**, **look-up**, **Benchmark** and **Syzygy** cells and paste it back.